##### Copyright 2026 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Extract music chords from audio with the Gemini API

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/examples/Audio_chord_extraction.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

This notebook shows you how to use the Gemini API's audio understanding capabilities to extract musical chord progressions from audio files. You will upload an audio file using the [File API](https://ai.google.dev/gemini-api/docs/file-prompting-strategies), prompt Gemini to analyze the music, and get back structured chord data in JSON format using [structured output](../quickstarts/JSON_mode.ipynb).

This example combines two powerful Gemini features:
- **Audio understanding**: Gemini can listen to audio files and understand musical content, including chords, tempo, and key.
- **JSON mode**: You can ask Gemini to return structured data, making it easy to process the results programmatically.

## Setup

### Install the SDK

In [ ]:
%pip install -U -q "google-genai>=1.0.0"

### Set up your API key

To run the following cell, your API key must be stored it in a Colab Secret named `GOOGLE_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see the [Authentication ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb) quickstart for an example.

In [ ]:
from google.colab import userdata
from google import genai

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
client = genai.Client(api_key=GOOGLE_API_KEY)

Select the model to use:

In [ ]:
MODEL_ID = "gemini-3-flash-preview" # @param ["gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.5-pro", "gemini-3.1-flash-lite-preview", "gemini-3-flash-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

## Upload an audio file

First, download a sample music file and upload it to the Gemini API using the [File API](https://ai.google.dev/gemini-api/docs/file-prompting-strategies). You can replace this URL with any publicly accessible audio file containing music.

In [ ]:
AUDIO_URL = "https://storage.googleapis.com/generativeai-downloads/data/State_of_the_Union_Address_30_January_1961.mp3"  # @param {type:"string"}

In [ ]:
!wget -q $AUDIO_URL -O sample.mp3

In [ ]:
audio_file = client.files.upload(file="sample.mp3")
print(f"Uploaded: {audio_file.name}")

## Extract chords with a simple prompt

Ask Gemini to listen to the audio and identify the chord progression. Use JSON mode to get structured output that you can easily process.

In [ ]:
import json

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        audio_file,
        """
        Analyze this audio file and extract any musical chord information.
        Return a JSON object with:
        - "title": the song title if identifiable, otherwise "Unknown"
        - "tempo": estimated tempo in BPM
        - "key": the musical key
        - "progression": the chord progression as a string (e.g. "Am - G - F - E")
        - "chords": a list of objects, each with "root_note", "chord_type",
          and "chord_notation"
        """,
    ],
    config={
        "response_mime_type": "application/json",
    },
)

chord_data = json.loads(response.text)
print(json.dumps(chord_data, indent=2))

## Use structured output with a schema

For more predictable results, define a schema using Python dataclasses and pass it to `response_schema`. This ensures the output always matches your expected format. See the [JSON mode quickstart](../quickstarts/JSON_mode.ipynb) for more details.

### Define the chord data model

In [ ]:
import dataclasses
import enum


class ChordType(enum.Enum):
    MAJOR = "major"
    MINOR = "minor"
    SEVENTH = "7"
    MAJOR_SEVENTH = "maj7"
    MINOR_SEVENTH = "m7"
    SUSPENDED_FOURTH = "sus4"
    SUSPENDED_SECOND = "sus2"
    DIMINISHED = "dim"
    AUGMENTED = "aug"


@dataclasses.dataclass
class Chord:
    root_note: str
    chord_type: ChordType
    chord_notation: str


@dataclasses.dataclass
class ChordProgression:
    title: str
    tempo: str
    key: str
    progression: str
    chords: list[Chord]

### Extract chords with the schema

Now pass the `ChordProgression` dataclass as the `response_schema`. Gemini will return data that matches the schema exactly.

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        audio_file,
        "Analyze this audio and extract the chord progression.",
    ],
    config={
        "response_mime_type": "application/json",
        "response_schema": ChordProgression,
    },
)

result = json.loads(response.text)
print(json.dumps(result, indent=2))

## Display the results

Print the extracted chord information in a human-readable format.

In [ ]:
print(f"Title: {result.get('title', 'Unknown')}")
print(f"Key: {result.get('key', 'Unknown')}")
print(f"Tempo: {result.get('tempo', 'Unknown')}")
print(f"Progression: {result.get('progression', 'Unknown')}")
print("\nChords:")
for chord in result.get("chords", []):
    notation = chord.get("chord_notation", "?")
    root = chord.get("root_note", "?")
    ctype = chord.get("chord_type", "?")
    print(f"  {notation}  (root: {root}, type: {ctype})")

## Use a YouTube video

You can also extract chords from a YouTube music video. Pass the URL directly as part of the request content using `types.Part`.

In [ ]:
from google.genai import types

youtube_url = "https://www.youtube.com/watch?v=dQw4w9WgXcQ"  # @param {type:"string"}

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        types.Part.from_uri(
            file_uri=youtube_url,
            mime_type="video/*",
        ),
        "Analyze the music in this video and extract the chord progression.",
    ],
    config={
        "response_mime_type": "application/json",
        "response_schema": ChordProgression,
    },
)

yt_result = json.loads(response.text)
print(json.dumps(yt_result, indent=2))

## Next steps

### Useful API references:

- Learn more about [audio understanding](https://ai.google.dev/gemini-api/docs/audio) in the Gemini API docs.
- See the [Audio quickstart ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Audio.ipynb) for more audio use cases.
- Check out the [JSON mode quickstart ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/JSON_mode.ipynb) for more on structured output.
- Explore the [File API quickstart ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/File_API.ipynb) for more on file uploads.

### Related examples:

- [Voice memos ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](Voice_memos.ipynb): Summarize voice memos with Gemini.